## Creating the Database

In [1]:
import pandas as pd
import sqlite3

# Read the cleaned Excel file
df = pd.read_excel(r"C:\Users\DELL\Desktop\Data Analytics Project\Swiggy Analysis\Swiggy_Data.xlsx")

# Create a new SQLite database
conn = sqlite3.connect(r"C:\Users\DELL\Desktop\Data Analytics Project\Swiggy Analysis\SwiggyDB_Clean.db")

# Write the DataFrame to SQLite
df.to_sql("Swiggy_Data", conn, if_exists="replace", index=False)

conn.close()

print("Database created successfully!")

Database created successfully!


In [2]:
conn = sqlite3.connect(r"C:\Users\DELL\Desktop\Data Analytics Project\Swiggy Analysis\SwiggyDB_Clean.db")

query = """
SELECT *
FROM Swiggy_Data
LIMIT 5;
"""

df = pd.read_sql(query, conn)

df

,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count
0,Karnataka,Bengaluru,29-06-2025,Anand Sweets & Savouries,Rajarajeshwari Nagar,Veg,Butter Murukku200gm,133.9,4.0,0
1,Karnataka,Bengaluru,03-04-2025,Srinidhi Sagar Deluxe,Kengeri,Veg,Badam Milk,52.0,4.5,25
2,Karnataka,Bengaluru,15-01-2025,Srinidhi Sagar Deluxe,Kengeri,Veg,Chow Chow Bath,117.0,4.7,48
3,Karnataka,Bengaluru,17-04-2025,Srinidhi Sagar Deluxe,Kengeri,Veg,Kesari Bath,65.0,4.6,65
4,Karnataka,Bengaluru,13-03-2025,Srinidhi Sagar Deluxe,Kengeri,Veg,Mix Raitha,130.0,4.0,0


In [3]:
query = """
PRAGMA table_info(Swiggy_Data);
"""

pd.read_sql(query, conn)

,cid,name,type,notnull,dflt_value,pk
0,0,State,TEXT,0,None,0
1,1,City,TEXT,0,None,0
2,2,Order Date,TEXT,0,None,0
3,3,Restaurant Name,TEXT,0,None,0
4,4,Location,TEXT,0,None,0
5,5,Category,TEXT,0,None,0
6,6,Dish Name,TEXT,0,None,0
7,7,Price (INR),REAL,0,None,0
8,8,Rating,REAL,0,None,0
9,9,Rating Count,INTEGER,0,None,0


## Cleaning Data 

In [4]:
query = """
SELECT
    SUM(CASE WHEN State IS NULL THEN 1 ELSE 0 END) AS null_state,
    SUM(CASE WHEN City IS NULL THEN 1 ELSE 0 END) AS null_city,
    SUM(CASE WHEN [Order Date] IS NULL THEN 1 ELSE 0 END) AS null_order_date,
    SUM(CASE WHEN [Restaurant Name] IS NULL THEN 1 ELSE 0 END) AS null_restaurant_name,
    SUM(CASE WHEN Location IS NULL THEN 1 ELSE 0 END) AS null_location,
    SUM(CASE WHEN Category IS NULL THEN 1 ELSE 0 END) AS null_category,
    SUM(CASE WHEN [Dish Name] IS NULL THEN 1 ELSE 0 END) AS null_dish_name,
    SUM(CASE WHEN [Price (INR)] IS NULL THEN 1 ELSE 0 END) AS null_price,
    SUM(CASE WHEN Rating IS NULL THEN 1 ELSE 0 END) AS null_rating,
    SUM(CASE WHEN [Rating Count] IS NULL THEN 1 ELSE 0 END) AS null_rating_count
FROM Swiggy_Data;
"""

df = pd.read_sql(query, conn)
df

,null_state,null_city,null_order_date,null_restaurant_name,null_location,null_category,null_dish_name,null_price,null_rating,null_rating_count
0,0,0,0,0,0,0,0,0,0,0


## Blank or Empty Strings

In [5]:
query=""" 
SELECT * FROM Swiggy_Data
Where
    State = ' ' OR City = ' ' OR [Restaurant Name] = ' ' OR Location = ' ' OR Category = ' ' OR [Dish Name] = ' ';
"""
df = pd.read_sql(query, conn)
df

,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count


## Finding Duplicate Data 

In [6]:
query=""" 
SELECT 
    State,City,[Order Date],[Restaurant Name],Location,Category,[Dish Name],[Price (INR)],Rating,[Rating Count],
    Count(*) AS cnt
From Swiggy_Data
Group By
State,City,[Order Date],[Restaurant Name],Location,Category,[Dish Name],[Price (INR)],Rating,[Rating Count]
    HAVING Count(*) > 1;

"""
df = pd.read_sql(query, conn)
df

,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count,cnt
0,Assam,Guwahati,10-05-2025,LunchBox - Meals and Thalis,Beltola,Veg,Dal Makhani Rice Lunchbox with Gulab Jamun 2 pcs,299.00,5.0,6,2
1,Assam,Guwahati,10-06-2025,Spice Garden,Maligaon,Veg,Butter Tawa Roti,30.00,3.8,5,2
2,Assam,Guwahati,11-08-2025,KFC,Borjhar,Non-Veg,Korean Tangy Chicken Roll,139.00,4.2,57,2
3,Assam,Guwahati,20-05-2025,Loyans Bakery & Confectionery,Azara,Veg,Chocolate Almond Cake GF,119.00,4.4,0,2
4,Assam,Guwahati,20-08-2025,Domino's Pizza,Jatia,Veg,Margherita Pizza,109.00,4.4,252,2
...,...,...,...,...,...,...,...,...,...,...,...
113,West Bengal,Kolkata,13-03-2025,Grameen Kulfi,Behala,Veg,Desi Malai Matka Kulfi pack Of 2,152.54,4.8,212,2
114,West Bengal,Kolkata,16-05-2025,Domino's Pizza,Barasat,Non-Veg,Non Veg Loaded,199.00,4.5,598,2
115,West Bengal,Kolkata,21-06-2025,Sweet Truth - Cake and Desserts,Madhyamgram,Veg,Mango Cheesecake,209.00,4.5,79,2
116,West Bengal,Kolkata,24-08-2025,BOOM - Sub Style Sandwiches,Chowringhee,Veg,Tikki Chaat Style SubSandwich,129.00,4.4,0,2


## Delete Duplicate Data

In [7]:
query = """
DELETE FROM Swiggy_Data
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM Swiggy_Data
    GROUP BY
        State,
        City,
        [Order Date],
        [Restaurant Name],
        Location,
        Category,
        [Dish Name],
        [Price (INR)],
        Rating,
        [Rating Count]
);
"""

conn.execute(query)
conn.commit()

print("Duplicate records removed successfully.")

Duplicate records removed successfully.


In [8]:
query = """
SELECT
    State,
    City,
    [Order Date],
    [Restaurant Name],
    Location,
    Category,
    [Dish Name],
    [Price (INR)],
    Rating,
    [Rating Count],
    COUNT(*) AS cnt
FROM Swiggy_Data
GROUP BY
    State,
    City,
    [Order Date],
    [Restaurant Name],
    Location,
    Category,
    [Dish Name],
    [Price (INR)],
    Rating,
    [Rating Count]
HAVING COUNT(*) > 1;
"""

df = pd.read_sql(query, conn)
df

,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count,cnt


# Create Schema

## Date Table

### Note: The following cells create and populate the data warehouse. They only need to be run once. If the tables already exist, you can skip these cells and proceed directly to the analysis section.

In [9]:
#dim_date table
query = """
CREATE TABLE IF NOT EXISTS dim_date (
    date_id INTEGER PRIMARY KEY AUTOINCREMENT,
    Full_Date DATE,
    Year INTEGER,
    Month INTEGER,
    Month_Name TEXT,
    Quarter INTEGER,
    Week INTEGER
);
"""

conn.execute(query)
conn.commit()

print("Table checked/created successfully!")

Table checked/created successfully!


## Order Date Modified

In [10]:
query = """
UPDATE Swiggy_Data
SET [Order Date] =
    substr([Order Date], 7, 4) || '-' ||
    substr([Order Date], 4, 2) || '-' ||
    substr([Order Date], 1, 2);
"""

conn.execute(query)
conn.commit()

print("Order Date format updated successfully!")

Order Date format updated successfully!


In [12]:
query = """
CREATE TABLE IF NOT EXISTS dim_date (
    date_id INTEGER PRIMARY KEY AUTOINCREMENT,
    Full_Date DATE,
    Year INTEGER,
    Month INTEGER,
    Month_Name TEXT,
    Quarter INTEGER,
    Week INTEGER
);
"""

conn.execute(query)
conn.commit()

print("Table checked/created successfully!")

Table checked/created successfully!


In [22]:
query = """
CREATE TABLE IF NOT EXISTS dim_location (
    location_id INTEGER PRIMARY KEY AUTOINCREMENT,
    State TEXT,
    City TEXT,
    Location TEXT
);
"""

conn.execute(query)
conn.commit()

print("dim_location checked/created successfully!")

dim_location checked/created successfully!


In [16]:
query = """
CREATE TABLE IF NOT EXISTS dim_restaurant (
    restaurant_id INTEGER PRIMARY KEY AUTOINCREMENT,
    Restaurant_Name TEXT
);
"""

conn.execute(query)
conn.commit()

print("dim_restaurant table created successfully!")

dim_restaurant table created successfully!


In [17]:
#dim_category
query = """
CREATE TABLE IF NOT EXISTS dim_category (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    Category TEXT
);
"""

conn.execute(query)
conn.commit()

print("dim_category table created successfully!")

dim_category table created successfully!


In [18]:
#dim_dish
query = """
CREATE TABLE IF NOT EXISTS dim_dish (
    dish_id INTEGER PRIMARY KEY AUTOINCREMENT,
    Dish_Name TEXT
);
"""

conn.execute(query)
conn.commit()

print("dim_dish table created successfully!")

dim_dish table created successfully!


In [19]:
#fact_swiggy_orders
query = """
CREATE TABLE IF NOT EXISTS fact_swiggy_orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,

    date_id INTEGER,
    Price_INR REAL,
    Rating REAL,
    Rating_Count INTEGER,

    location_id INTEGER,
    restaurant_id INTEGER,
    category_id INTEGER,
    dish_id INTEGER,

    FOREIGN KEY(date_id) REFERENCES dim_date(date_id),
    FOREIGN KEY(location_id) REFERENCES dim_location(location_id),
    FOREIGN KEY(restaurant_id) REFERENCES dim_restaurant(restaurant_id),
    FOREIGN KEY(category_id) REFERENCES dim_category(category_id),
    FOREIGN KEY(dish_id) REFERENCES dim_dish(dish_id)
);
"""

conn.execute(query)
conn.commit()

print("fact_swiggy_orders table created successfully!")

fact_swiggy_orders table created successfully!


In [20]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

df = pd.read_sql(query, conn)
df

,name
0,dim_date
1,sqlite_sequence
2,dim_restaurant
3,dim_category
4,dim_dish
5,fact_swiggy_orders
6,dim_location
7,Swiggy_Data


## Insert Data in all Tables

In [49]:
query = """
INSERT OR IGNORE INTO dim_date
(Full_Date, Year, Month, Month_Name, Quarter, Day, Week)

SELECT DISTINCT
    [Order Date],
    CAST(strftime('%Y', [Order Date]) AS INTEGER),
    CAST(strftime('%m', [Order Date]) AS INTEGER),

    CASE strftime('%m', [Order Date])
        WHEN '01' THEN 'January'
        WHEN '02' THEN 'February'
        WHEN '03' THEN 'March'
        WHEN '04' THEN 'April'
        WHEN '05' THEN 'May'
        WHEN '06' THEN 'June'
        WHEN '07' THEN 'July'
        WHEN '08' THEN 'August'
        WHEN '09' THEN 'September'
        WHEN '10' THEN 'October'
        WHEN '11' THEN 'November'
        WHEN '12' THEN 'December'
    END,

    ((CAST(strftime('%m', [Order Date]) AS INTEGER)-1)/3)+1,

    CAST(strftime('%d', [Order Date]) AS INTEGER),

    CAST(strftime('%W', [Order Date]) AS INTEGER)

FROM Swiggy_Data
WHERE [Order Date] IS NOT NULL;
"""

conn.execute(query)
conn.commit()

print("dim_date loaded successfully.")

dim_date loaded successfully.


In [36]:
query = """
SELECT COUNT(*) AS Total_Dates
FROM dim_date;
"""

pd.read_sql(query, conn)

,Total_Dates
0,243


In [50]:
query = """
SELECT *
FROM dim_date
LIMIT 10;
"""

df = pd.read_sql(query, conn)
df

,date_id,Full_Date,Year,Month,Month_Name,Quarter,Day,Week
0,730,2025-06-29,2025,6,June,2,29,25
1,731,2025-04-03,2025,4,April,2,3,13
2,732,2025-01-15,2025,1,January,1,15,2
3,733,2025-04-17,2025,4,April,2,17,15
4,734,2025-03-13,2025,3,March,1,13,10
5,735,2025-07-08,2025,7,July,3,8,27
6,736,2025-01-21,2025,1,January,1,21,3
7,737,2025-04-13,2025,4,April,2,13,14
8,738,2025-05-02,2025,5,May,2,2,17
9,739,2025-07-30,2025,7,July,3,30,30


In [51]:
#dim_location
query = """
INSERT INTO dim_location(State, City, Location)

SELECT DISTINCT
State,
City,
Location

FROM Swiggy_Data;
"""

conn.execute(query)
conn.commit()
print("dim_location populated successfully!")

dim_location populated successfully!


In [52]:
query = """
SELECT *
FROM dim_location
LIMIT 10;
"""

df = pd.read_sql(query, conn)
df

,location_id,State,City,Location
0,2986,Karnataka,Bengaluru,Rajarajeshwari Nagar
1,2987,Karnataka,Bengaluru,Kengeri
2,2988,Karnataka,Bengaluru,Kanmanike Village
3,2989,Karnataka,Bengaluru,Bidadi
4,2990,Karnataka,Bengaluru,Kengeri Satellite Town
5,2991,Karnataka,Bengaluru,Kanakapura Road
6,2992,Karnataka,Bengaluru,Jawaharlal Nehru Road
7,2993,Karnataka,Bengaluru,KENGERI UPANAGAR
8,2994,Karnataka,Bengaluru,Rr Nagar
9,2995,Karnataka,Bengaluru,Gopalan Arcade Mall


In [53]:
#dim_restaurant
query = """
INSERT INTO dim_restaurant (Restaurant_Name)

SELECT DISTINCT
    [Restaurant Name]
FROM Swiggy_Data
WHERE [Restaurant Name] IS NOT NULL;
"""

conn.execute(query)
conn.commit()

print("dim_restaurant populated successfully!")

dim_restaurant populated successfully!


In [54]:
query = """
SELECT *
FROM dim_restaurant
LIMIT 10;
"""

df = pd.read_sql(query, conn)
df

,restaurant_id,Restaurant_Name
0,1987,Anand Sweets & Savouries
1,1988,Srinidhi Sagar Deluxe
2,1989,Thalassery Restaurant
3,1990,Appu Donne Biriyani Palace
4,1991,Bismillah Taj Hotel
5,1992,Nisarga Family Daba
6,1993,Swadh Restaurant
7,1994,Sri Vinayaka Hot Chips
8,1995,Chinese Wok
9,1996,Pizza Hut


In [55]:
#dim_category
query = """
INSERT INTO dim_category (Category)

SELECT DISTINCT
    Category
FROM Swiggy_Data
WHERE Category IS NOT NULL;
"""

conn.execute(query)
conn.commit()

print("dim_category populated successfully!")

dim_category populated successfully!


In [56]:
query = """
SELECT *
FROM dim_category;
"""

df = pd.read_sql(query, conn)
df

,category_id,Category
0,5,Veg
1,6,Non-Veg


In [57]:
#dim_dish
query = """
INSERT INTO dim_dish (Dish_Name)

SELECT DISTINCT
    [Dish Name]
FROM Swiggy_Data
WHERE [Dish Name] IS NOT NULL;
"""

conn.execute(query)
conn.commit()

print("dim_dish populated successfully!")

dim_dish populated successfully!


In [58]:
query = """
SELECT *
FROM dim_dish
LIMIT 10;
"""

df = pd.read_sql(query, conn)
df

,dish_id,Dish_Name
0,110427,Butter Murukku200gm
1,110428,Badam Milk
2,110429,Chow Chow Bath
3,110430,Kesari Bath
4,110431,Mix Raitha
5,110432,Srinidhi Sagar Special
6,110433,Garlic Naan
7,110434,Pista
8,110435,Panneer Butter Masala
9,110436,Dal Tadka


In [59]:
query = """
INSERT INTO fact_swiggy_orders
(
    date_id,
    Price_INR,
    Rating,
    Rating_Count,
    location_id,
    restaurant_id,
    category_id,
    dish_id
)

SELECT
    dd.date_id,
    s.[Price (INR)],
    s.Rating,
    s.[Rating Count],
    dl.location_id,
    dr.restaurant_id,
    dc.category_id,
    dsh.dish_id

FROM Swiggy_Data s

JOIN dim_date dd
    ON dd.Full_Date = s.[Order Date]

JOIN dim_location dl
    ON dl.State = s.State
    AND dl.City = s.City
    AND dl.Location = s.Location

JOIN dim_restaurant dr
    ON  dr.Restaurant_Name = s.[Restaurant Name]

JOIN dim_category dc
    ON dc.Category = s.Category  

INNER JOIN dim_dish dsh
    ON dsh.Dish_Name = s.[Dish Name];
"""

conn.execute(query)
conn.commit()

print("Fact table populated successfully!")

Fact table populated successfully!


In [60]:
query = """
SELECT *
FROM fact_swiggy_orders
LIMIT 10;
"""

df = pd.read_sql(query, conn)
df

,order_id,date_id,Price_INR,Rating,Rating_Count,location_id,restaurant_id,category_id,dish_id
0,5524737,730,133.9,4.0,0,2986,1987,5,110427
1,5524738,731,52.0,4.5,25,2987,1988,5,110428
2,5524739,732,117.0,4.7,48,2987,1988,5,110429
3,5524740,733,65.0,4.6,65,2987,1988,5,110430
4,5524741,734,130.0,4.0,0,2987,1988,5,110431
5,5524742,735,312.0,4.0,0,2987,1988,5,110432
6,5524743,736,98.0,4.0,34,2987,1988,5,110433
7,5524744,737,137.0,4.0,0,2987,1988,5,110434
8,5524745,738,241.0,4.4,29,2987,1988,5,110435
9,5524746,739,195.0,4.9,51,2987,1988,5,110436


In [86]:
query = """
SELECT
    f.order_id,
    d.Full_Date,
    d.Month_Name,
    d.Year,

    l.State,
    l.City,
    l.Location,

    r.Restaurant_Name,

    c.Category,

    di.Dish_Name,

    f.Price_INR,
    f.Rating,
    f.Rating_Count

FROM fact_swiggy_orders f

JOIN dim_date d
    ON f.date_id = d.date_id

JOIN dim_location l
    ON f.location_id = l.location_id

JOIN dim_restaurant r
    ON f.restaurant_id = r.restaurant_id

JOIN dim_category c
    ON f.category_id = c.category_id

JOIN dim_dish di
    ON f.dish_id = di.dish_id;
"""

df = pd.read_sql(query, conn)
df

,order_id,Full_Date,Month_Name,Year,State,City,Location,Restaurant_Name,Category,Dish_Name,Price_INR,Rating,Rating_Count
0,5524737,2025-06-29,June,2025,Karnataka,Bengaluru,Rajarajeshwari Nagar,Anand Sweets & Savouries,Veg,Butter Murukku200gm,133.9,4.0,0
1,5524738,2025-04-03,April,2025,Karnataka,Bengaluru,Kengeri,Srinidhi Sagar Deluxe,Veg,Badam Milk,52.0,4.5,25
2,5524739,2025-01-15,January,2025,Karnataka,Bengaluru,Kengeri,Srinidhi Sagar Deluxe,Veg,Chow Chow Bath,117.0,4.7,48
3,5524740,2025-04-17,April,2025,Karnataka,Bengaluru,Kengeri,Srinidhi Sagar Deluxe,Veg,Kesari Bath,65.0,4.6,65
4,5524741,2025-03-13,March,2025,Karnataka,Bengaluru,Kengeri,Srinidhi Sagar Deluxe,Veg,Mix Raitha,130.0,4.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
197307,5722044,2025-01-25,January,2025,Sikkim,Gangtok,Gangtok,Mama's Kitchen,Veg,Soya cheese chilli momo,112.0,4.4,0
197308,5722045,2025-07-02,July,2025,Sikkim,Gangtok,Gangtok,Mama's Kitchen,Veg,Kurkure momo fried,140.0,4.4,0
197309,5722046,2025-03-25,March,2025,Sikkim,Gangtok,Gangtok,Mama's Kitchen,Veg,Chilli cheese momo,126.0,4.4,0
197310,5722047,2025-03-26,March,2025,Sikkim,Gangtok,Gangtok,Mama's Kitchen,Veg,Veg Momos 8 Pc,85.0,4.4,0


## KPI's

## Total Orders

In [70]:
query = """ 
Select COUNT(*) AS Total_Orders
FROM fact_swiggy_orders
"""
pd.read_sql(query,conn)

,Total_Orders
0,197312


## Total Revenue(INR)

In [71]:
query = """ 
SELECT SUM(Price_INR) from fact_swiggy_orders
"""
pd.read_sql(query,conn)

,SUM(Price_INR)
0,52983414.11


## Average Order Value

In [72]:
query = """
SELECT
    ROUND(AVG(Price_INR),2) AS Average_Order_Value
FROM fact_swiggy_orders;
"""

pd.read_sql(query, conn)

,Average_Order_Value
0,268.53


## Monthly Revenue Trend

In [73]:
query = """
SELECT
    d.Month_Name,
    d.Month,
    ROUND(SUM(f.Price_INR),2) AS Revenue

FROM fact_swiggy_orders f

JOIN dim_date d
ON f.date_id = d.date_id

GROUP BY
    d.Month,
    d.Month_Name

ORDER BY d.Month;
"""

pd.read_sql(query, conn)

,Month_Name,Month,Revenue
0,January,1,6821195.06
1,February,2,6262807.83
2,March,3,6570614.33
3,April,4,6587674.00
4,May,5,6790616.35
5,June,6,6512389.19
6,July,7,6649218.13
7,August,8,6788899.22


## Quarterly Revenue Trend


In [74]:
query = """
SELECT
    d.Quarter,
    ROUND(SUM(f.Price_INR),2) AS Revenue

FROM fact_swiggy_orders f

JOIN dim_date d
ON f.date_id = d.date_id

GROUP BY d.Quarter

ORDER BY d.Quarter;
"""

pd.read_sql(query, conn)

,Quarter,Revenue
0,1,19654617.22
1,2,19890679.54
2,3,13438117.35


## Revenue by State

In [75]:
query = """
SELECT
    l.State,
    ROUND(SUM(f.Price_INR),2) AS Revenue

FROM fact_swiggy_orders f

JOIN dim_location l
ON f.location_id = l.location_id

GROUP BY l.State

ORDER BY Revenue DESC;
"""

pd.read_sql(query, conn)

,State,Revenue
0,Karnataka,5450921.59
1,Uttar Pradesh,3116332.13
2,Telangana,3021176.35
3,Maharashtra,3014039.35
4,Delhi,2828122.33
5,Gujarat,2813666.94
6,Punjab,2804931.82
7,West Bengal,2661339.22
8,Tamil Nadu,2642102.63
9,Rajasthan,2502044.32


## Revenue by City

In [76]:
query = """
SELECT
    l.City,
    ROUND(SUM(f.Price_INR),2) AS Revenue

FROM fact_swiggy_orders f

JOIN dim_location l
ON f.location_id = l.location_id

GROUP BY l.City

ORDER BY Revenue DESC;
"""

pd.read_sql(query, conn)

,City,Revenue
0,Bengaluru,5450921.59
1,Lucknow,3116332.13
2,Hyderabad,3021176.35
3,Mumbai,3014039.35
4,New Delhi,2828122.33
5,Ahmedabad,2813666.94
6,Chandigarh,2804931.82
7,Kolkata,2661339.22
8,Chennai,2642102.63
9,Jaipur,2502044.32


## Top 10 Restaurants by Revenue

In [77]:
query = """
SELECT
    r.Restaurant_Name,
    ROUND(SUM(f.Price_INR),2) AS Revenue

FROM fact_swiggy_orders f

JOIN dim_restaurant r
ON f.restaurant_id = r.restaurant_id

GROUP BY r.Restaurant_Name

ORDER BY Revenue DESC

LIMIT 10;
"""

pd.read_sql(query, conn)

,Restaurant_Name,Revenue
0,KFC,4243867.78
1,McDonald's,3340012.82
2,Pizza Hut,2131929.69
3,Burger King,1899920.09
4,Domino's Pizza,1831831.32
5,Olio - The Wood Fired Pizzeria,1232731.00
6,LunchBox - Meals and Thalis,1100384.00
7,Baskin Robbins - Ice Cream Desserts,860515.67
8,"Faasos - Wraps, Rolls & Shawarma",778910.00
9,The Good Bowl,672655.00


## Top 10 Dishes by Revenue

In [78]:
query = """
SELECT
    d.Dish_Name,
    ROUND(SUM(f.Price_INR),2) AS Revenue

FROM fact_swiggy_orders f

JOIN dim_dish d
ON f.dish_id = d.dish_id

GROUP BY d.Dish_Name

ORDER BY Revenue DESC

LIMIT 10;
"""

pd.read_sql(query, conn)

,Dish_Name,Revenue
0,Bold BBQ Veggie Thin n Crispy,99617.00
1,Korean Thai Roll Chicken Meal,95591.57
2,Full House Popcorn Chicken Bucket,79200.00
3,Paneer Butter Masala,70713.82
4,Indian Tandoori Roll Chicken Meal,70392.05
5,Triple Chicken Feast,69939.00
6,Big 12 Chicken Bucket,69738.12
7,Veggie Supreme,67841.00
8,Hot Crispy Chicken 8 pcs,66560.85
9,Ultimate Savings Chicken Bucket,64611.68


## Veg vs Non-Veg Revenue

In [79]:
query = """
SELECT
    c.Category,
    ROUND(SUM(f.Price_INR),2) AS Revenue

FROM fact_swiggy_orders f

JOIN dim_category c
ON f.category_id = c.category_id

GROUP BY c.Category;
"""

pd.read_sql(query, conn)

,Category,Revenue
0,Non-Veg,19235151.05
1,Veg,33748263.06


## Average Rating by Restaurant

In [80]:
query = """
SELECT
    r.Restaurant_Name,
    ROUND(AVG(f.Rating),2) AS Average_Rating

FROM fact_swiggy_orders f

JOIN dim_restaurant r
ON f.restaurant_id = r.restaurant_id

GROUP BY r.Restaurant_Name

ORDER BY Average_Rating DESC

LIMIT 10;
"""

pd.read_sql(query, conn)

,Restaurant_Name,Average_Rating
0,Sakana,4.82
1,Vijay Dairy,4.75
2,Jagannath Mandir Arna Prasad,4.75
3,Radhey Lal's Parampara Sweets,4.74
4,I Deli Cafe,4.74
5,Yadav Doodh Dairy,4.73
6,Tossin Pizza,4.71
7,Perambur Sri Srinivasa Sweets & Snacks,4.71
8,Wooden Plate,4.69
9,Natural Ice Cream,4.68


## Top 10 Cities by Orders

In [81]:
query = """
SELECT
    l.City,
    COUNT(*) AS Total_Orders

FROM fact_swiggy_orders f

JOIN dim_location l
ON f.location_id = l.location_id

GROUP BY l.City

ORDER BY Total_Orders DESC

LIMIT 10;
"""

pd.read_sql(query, conn)

,City,Total_Orders
0,Bengaluru,20051
1,Mumbai,10505
2,Hyderabad,10304
3,Jaipur,10281
4,Lucknow,10188
5,New Delhi,10186
6,Ahmedabad,10168
7,Chandigarh,10059
8,Chennai,10040
9,Kolkata,10038


## Highest Rated Dishes

In [82]:
query = """
SELECT
    d.Dish_Name,
    ROUND(AVG(f.Rating),2) AS Rating

FROM fact_swiggy_orders f

JOIN dim_dish d
ON f.dish_id = d.dish_id

GROUP BY d.Dish_Name

HAVING COUNT(*) > 20

ORDER BY Rating DESC

LIMIT 10;
"""

pd.read_sql(query, conn)

,Dish_Name,Rating
0,Madagascar Chocolate Ice Cream 500ml,4.83
1,Chicken Tikka 8 Pcs,4.73
2,Mango Cheesecake Box of 2,4.72
3,Tender Coconut Ice Cream 500ml,4.70
4,Mint Milk Chocolate Chips Ice Cream,4.68
5,Kesar Peda,4.68
6,Kaju Katli,4.68
7,Fruit Overload Ice Cream,4.68
8,Alphonso Mango Ice Cream 500ml,4.68
9,20 Pc Chicken Nuggets,4.67


## Top Restaurants by Orders

In [83]:
query = """
SELECT
    r.Restaurant_Name,
    COUNT(*) AS Orders

FROM fact_swiggy_orders f

JOIN dim_restaurant r
ON f.restaurant_id = r.restaurant_id

GROUP BY r.Restaurant_Name

ORDER BY Orders DESC

LIMIT 10;
"""

pd.read_sql(query, conn)

,Restaurant_Name,Orders
0,McDonald's,13517
1,KFC,12951
2,Burger King,7113
3,Pizza Hut,6525
4,Domino's Pizza,5485
5,LunchBox - Meals and Thalis,4697
6,Baskin Robbins - Ice Cream Desserts,4196
7,"Faasos - Wraps, Rolls & Shawarma",3251
8,Olio - The Wood Fired Pizzeria,3239
9,The Good Bowl,2663


## Average Revenue per State

In [84]:
query = """
SELECT
    l.State,
    ROUND(AVG(f.Price_INR),2) AS Average_Order_Value

FROM fact_swiggy_orders f

JOIN dim_location l
ON f.location_id = l.location_id

GROUP BY l.State

ORDER BY Average_Order_Value DESC;
"""

pd.read_sql(query, conn)

,State,Average_Order_Value
0,Goa,306.06
1,Uttar Pradesh,305.88
2,Telangana,293.20
3,Haryana,287.43
4,Maharashtra,286.91
5,Punjab,278.85
6,Delhi,277.65
7,Gujarat,276.72
8,Meghalaya,275.50
9,Himachal Pradesh,273.57
